In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL

import openmm.app as app
from ase.io import read

In [2]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/water_refit.xml')
ff = ForceFieldXML(ff_path, device=device)



In [3]:
water_cluster_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')

In [4]:
water_cluster_pdb = app.PDBFile(water_cluster_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_fd_morse=False, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [8]:
coords = torch.from_numpy(positions[-1] / BOHR2ANG).to(device).requires_grad_(False)
natoms = coords.shape[-1]
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
energies = systems[-1].getEnergy(coords, box)
print(energies)
print(energies['total'] * HARTREE2KCAL)

{'bond': tensor(0.0050), 'angle': tensor(0.0010), 'torsion': tensor(0.), 'bond_bond': tensor(-0.0003), 'bond_angle': tensor(0.0005), 'angle_angle': tensor(0.), 'torsion_bond': tensor(0.), 'torsion_angle': tensor(0.), 'torsion_angle_angle': tensor(0.), 'perm_elec': tensor(-0.8076), 'pol': tensor(-0.2094), 'ct_direct': tensor(-0.2435), 'xpol': tensor(-0.0103), 'pauli': tensor(0.9493), 'disp': tensor(-0.1932), 'total': tensor(-0.5083)}
tensor(-318.9818)
